### A.5 实现多层神经网络
#### 代码清单 A-4 一个具有两个隐藏层的多层感知机

In [5]:
import torch

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs): # 将输入和输出的数量编码为变量，使我们可以在具有不同特征数量和类别数量的数据集上重复相同的代码
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 第一个隐藏层
            torch.nn.Linear(num_inputs, 30), # 线性层将输入结点和输出结点的数量作为参数
            torch.nn.ReLU(), # 非线性激活函数被放置在隐藏层之间

            # 第二个隐藏层
            torch.nn.Linear(30, 20), # 下一个隐藏层的输出节点数量必须与下一层的输入节点数量相匹配
            torch.nn.ReLU(),

            # 输出层
            torch.nn.Linear(20, num_outputs)
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits # 最后一层的输出称为 logits

model = NeuralNetwork(50, 3)
print(model) # 查看模型结构的摘要

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [6]:
# 检查模型的可训练参数总数
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total number of trainable model parameters: {num_params}')

Total number of trainable model parameters: 2213


In [7]:
# 访问对应的权重参数矩阵
print(model.layers[0].weight) # 访问第一个隐藏层的权重参数矩阵

Parameter containing:
tensor([[-0.0985, -0.1155, -0.0126,  ..., -0.1059,  0.0344,  0.0274],
        [ 0.0092,  0.1247,  0.0572,  ..., -0.1380, -0.0628,  0.1223],
        [ 0.1153, -0.0169, -0.0367,  ...,  0.0833,  0.1402,  0.1165],
        ...,
        [-0.0475, -0.0106, -0.1319,  ...,  0.0777, -0.0633,  0.0122],
        [ 0.0496,  0.1183, -0.0446,  ..., -0.1060,  0.0677, -0.1007],
        [ 0.0443,  0.0601, -0.1173,  ..., -0.0405, -0.1215,  0.1182]],
       requires_grad=True)


In [8]:
# 使用 .shape 属性查看其维度
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [9]:
# 访问对应的偏置参数向量
print(model.layers[0].bias) # 访问第一个隐藏层的偏置参数

Parameter containing:
tensor([ 0.0739, -0.1032, -0.0429,  0.0293,  0.0825, -0.0647,  0.0459,  0.0114,
        -0.0064,  0.0733, -0.0780,  0.0086, -0.0290,  0.0357, -0.1047, -0.0822,
         0.0264, -0.1298, -0.1145,  0.1022,  0.0700, -0.1128,  0.0187,  0.0725,
        -0.0781, -0.0015, -0.0750, -0.0154, -0.0140, -0.1064],
       requires_grad=True)


In [10]:
# 使用 .manual_seed() 函数设置随机数种子，从而确保每次运行代码时，结果都相同
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print( model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [11]:
# 通过前向传播使用 NeuralNetwork 实例
torch.manual_seed(123)
x = torch.rand(1, 50)
out = model(x)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [12]:
 ##### 输出的结果 tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)
 ##### grad_fn= ... 详见 grad_fn对照表.md

In [13]:
# 使用 torch.no_grad() 函数来禁用梯度计算
with torch.no_grad():
    out = model(x)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [14]:
# 为预测结果计算类别成员概率，就需要显示调用 softmax 函数
with torch.no_grad():
    out = torch.softmax(model(x), dim=1) # 通过指定 dim=1 来沿着正确的维度计算 softmax; dim=0 是批次维度，dim=1 是类别维度
print(out)

tensor([[0.3113, 0.3934, 0.2952]])
